# Import Libraries

In [49]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
import joblib

# Load Dataset

In [2]:
df = pd.read_csv(r"C:\Users\HP\Downloads\archive (4)\Titanic-Dataset.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


# Select Features and Target

In [4]:
X = df[["Pclass","Sex","Age","Fare","Embarked"]]
y = df["Survived"]

In [5]:
X.columns

Index(['Pclass', 'Sex', 'Age', 'Fare', 'Embarked'], dtype='object')

# Train-Test Split

In [6]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)

# Define Preprocessing Pipelines

In [7]:
numeric_features = ["Age","Fare"]
categorical_features = ["Pclass","Sex","Embarked"]
num_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler()),
    ("power",PowerTransformer(method="yeo-johnson"))
])
cat_pipeline = Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown = "ignore"))
])
preprocessor = ColumnTransformer([
    ("num",num_pipeline,numeric_features),
    ("cat",cat_pipeline,categorical_features)
])

# Create full pipeline (Preprocessing + Model)

In [81]:
pipeline = Pipeline([
    ("preprocessing",preprocessor),
    ("model",LogisticRegression(max_iter=200))
])

# Train the Pipeline

In [82]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler()),
                                                                  ('power',
                                                                   PowerTransformer())]),
                                                  ['Age', 'Fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Pclass', 'Sex',
                                                   'Embarked'])])),
                ('model', LogisticRegression(max_iter=200))])

# Evaluate Model

In [83]:
y_train_pred = pipeline.predict(X_train)
print("Training Accuracy:",accuracy_score(y_train,y_train_pred)*100)

Training Accuracy: 80.83832335329342


In [84]:
y_pred = pipeline.predict(X_test)
print("Accuracy:",accuracy_score(y_test,y_pred)*100)

Accuracy: 75.7847533632287


In [85]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.80      0.81      0.80       137
           1       0.69      0.67      0.68        86

    accuracy                           0.76       223
   macro avg       0.74      0.74      0.74       223
weighted avg       0.76      0.76      0.76       223



In [86]:
scores = cross_val_score(pipeline,X,y)
scores

array([0.77094972, 0.81460674, 0.79213483, 0.7752809 , 0.82022472])

In [87]:
format(scores.mean()*100,".2f")

'79.46'

# Save Full Pipeline (Deployment Ready)

In [88]:
import pickle
with open("titanic_pipeline.pkl","wb") as f:
    pickle.dump(pipeline,f)

In [89]:
joblib.dump(pipeline,"titanic_pipeline.pkl")
print("Pipeline saved in pkl file")

Pipeline saved in pkl file


# Load Pipeline and Predict (Deployment Simulation)

In [90]:
loaded_pipeline = joblib.load("titanic_pipeline.pkl")
new_passenger = pd.DataFrame({
    "Pclass":[1],
    "Sex":["Female"],
    "Age":[38],
    "Fare":[71.2833],
    "Embarked":["C"]
})
prediction = loaded_pipeline.predict(new_passenger)
print("Survived:",prediction[0])

Survived: 1
